# 34MLS — Assignment 3: Flow Field Through Porous Media

**Student name:** [Your name here]  
**Student ID:** [Your ID here]  
**Kaggle ID:** [Your Kaggle username here]  
**Deadline:** 11-04-2026 23:59

---

## About this notebook

This notebook is the appendix to the report `assignment3.tex`. It contains the full implementation code, structured in the same order as the report sections.

**Rubric map:**
- Step 1 (this section): Data exploration → feeds report **Section A** (dataset characterization)
- Steps 2–4: Preprocessing, augmentation → feeds report **Section A** (loss functions) and **Section B**
- Steps 5–12: Model implementation, training, comparisons → feeds report **Section C**
- Step 13: Kaggle submission → **Section D**

## Reproducibility seeds

We set seeds at the very top so that every random operation in this notebook — data splits, weight initialization, augmentation — produces the same result every time. This is required by **rubric E** (a peer must be able to reproduce our results).

- `torch.manual_seed` controls PyTorch random operations (weight init, dropout)
- `np.random.seed` controls NumPy random operations (shuffling, splits)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import os, json, time

# Reproducibility — must be set before any random operation
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device — use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Paths
DATA_DIR = 'flow-field-through-a-porous-media-25-26'
os.makedirs('experiments/logs',        exist_ok=True)
os.makedirs('experiments/plots',       exist_ok=True)
os.makedirs('experiments/checkpoints', exist_ok=True)
os.makedirs('submissions',             exist_ok=True)
os.makedirs('fig',                     exist_ok=True)

---
## Step 1 — Data Exploration

**What this step does:** Load all three training files and visualise the data before touching any model.

**Why before anything else?** Report section A requires us to *characterise the dataset in statistical terms* (rubric A, sub-point 3). We cannot write about symmetries or dimensional analysis without first having *seen* what the cross-sections and flow fields look like. This step produces the figures and numbers that will go directly into the report.

**The three files we load:**

| File | What it contains | Shape |
|------|-----------------|-------|
| `train_inputs.npy` | Binary cross-sections: 1 = open pixel, 0 = closed pixel | [1500, 32, 64] |
| `train_params.csv` | Physical parameters per sample: ΔP, L, μ, ΔA | [1500, 5 cols] |
| `train_labels.npy` | Ground truth flow rate field q(x,y) [m/s] | [1500, 32, 64] |

In [ ]:
# Load all training data
train_inputs = np.load(f'{DATA_DIR}/train_inputs.npy')      # binary cross-sections
train_labels = np.load(f'{DATA_DIR}/train_labels.npy')      # flow rate fields q(x,y)
train_params = pd.read_csv(f'{DATA_DIR}/train_params.csv')  # physical parameters

print("=== Shapes ===")
print(f"train_inputs : {train_inputs.shape}  dtype={train_inputs.dtype}")
print(f"train_labels : {train_labels.shape}  dtype={train_labels.dtype}")
print(f"train_params : {train_params.shape}")
print()
print("=== Parameter columns ===")
print(train_params.head())
print()
print("=== Input pixel values (should be 0 and 1 only) ===")
print(f"Unique values in inputs: {np.unique(train_inputs)}")
print()
print("=== Flow field range ===")
print(f"q min:  {train_labels.min():.4e} m/s")
print(f"q max:  {train_labels.max():.4e} m/s")
print(f"q mean: {train_labels.mean():.4e} m/s")

### 1.1 — Example cross-sections and flow fields

**What we plot:** 6 randomly chosen samples. For each: the binary cross-section (left) and the corresponding flow field q(x,y) (right).

**What to look for:**
- The flow field is zero wherever the pixel is closed (=0). You cannot have flow through a wall — this is a hard physical constraint our model must respect.
- The flow field is symmetric in a specific way: if the cross-section has a left-right symmetry, so does q(x,y). This is the equivariance we will discuss in report section A.
- The magnitude of q varies across the open region — pixels near the centre of an open channel flow faster than pixels near the edges (boundary layer effect).

In [ ]:
np.random.seed(SEED)  # ensure same samples every run
sample_ids = np.random.choice(len(train_inputs), size=6, replace=False)

fig, axes = plt.subplots(6, 2, figsize=(10, 18))
fig.suptitle('Example cross-sections and flow fields', fontsize=14, y=1.01)

for row, idx in enumerate(sample_ids):
    # Cross-section (binary image)
    ax_cs = axes[row, 0]
    im0 = ax_cs.imshow(train_inputs[idx], cmap='gray', vmin=0, vmax=1, aspect='auto')
    ax_cs.set_title(f'Sample {idx} — Cross-section')
    ax_cs.set_xlabel('y pixel index')
    ax_cs.set_ylabel('x pixel index')
    plt.colorbar(im0, ax=ax_cs, label='open (1) / closed (0)')

    # Flow field
    ax_fl = axes[row, 1]
    im1 = ax_fl.imshow(train_labels[idx], cmap='viridis', aspect='auto')
    ax_fl.set_title(f'Sample {idx} — Flow field q(x,y) [m/s]')
    ax_fl.set_xlabel('y pixel index')
    ax_fl.set_ylabel('x pixel index')
    plt.colorbar(im1, ax=ax_fl, label='q [m/s]')

plt.tight_layout()
plt.savefig('experiments/plots/exploration_cross_sections_and_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_cross_sections_and_flow.png")

### 1.2 — Physical parameter distributions

**What we plot:** Histograms of the four physical parameters across all 1500 training samples.

**Why this matters for the report:**
- Report section A asks us to *characterise the dataset in statistical terms*.
- The parameter distributions tell us how diverse the training set is — are we training on a narrow range of viscosities, or a wide one?
- The distributions also inform **dimensional analysis**: if ΔP, L, μ, ΔA all vary independently, we must make our model robust to all combinations. A model that hard-codes a specific viscosity will fail on the test set.

**Physical reminder of what each parameter means:**
- **ΔP [Pa]** — pressure drop driving the flow. Higher ΔP → faster flow. q scales linearly with ΔP (Darcy's law).
- **L [m]** — channel length. Longer channel → slower flow (more resistance). q scales as 1/L.
- **μ [Pa·s]** — fluid viscosity. More viscous fluid → slower flow. q scales as 1/μ.
- **ΔA [m²]** — pixel area. Affects the physical size of each open region.

In [ ]:
param_cols = ['delta_p', 'L', 'visc', 'delta_A']
param_labels = [
    r'$\Delta P$ [Pa]',
    r'$L$ [m]',
    r'$\mu$ [Pa$\cdot$s]',
    r'$\Delta A$ [m$^2$]'
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribution of physical parameters across 1500 training samples', fontsize=13)

for ax, col, label in zip(axes.flat, param_cols, param_labels):
    values = train_params[col].values
    ax.hist(values, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Count')
    ax.set_title(f'{label}  |  min={values.min():.3e}  max={values.max():.3e}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_parameter_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_parameter_distributions.png")

print("\n=== Parameter summary statistics ===")
print(train_params[param_cols].describe().to_string())

### 1.3 — Flow field value distribution

**What we plot:**
1. Histogram of ALL q values across all 1500 samples (one value per pixel per sample = 1500 × 32 × 64 = 3,072,000 values).
2. Distribution of the **total volumetric flow** Q per sample, where Q = Σ q(x,y) · ΔA [m³/s].

**Why:**
- The q histogram will show a large spike at zero (all closed pixels have q=0). The non-zero values tell us the actual flow rate magnitudes we need to predict.
- Q per sample tells us how much the total flow varies across the dataset — this matters for choosing a good loss function.
- These plots go directly into the report (rubric A, dataset characterisation).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Flow field statistics across training set', fontsize=13)

# --- Plot 1: all q values (log scale to show the zero spike) ---
ax = axes[0]
all_q = train_labels.flatten()
ax.hist(all_q, bins=80, color='steelblue', edgecolor='none', log=True)
ax.set_xlabel('q(x,y) [m/s]')
ax.set_ylabel('Count (log scale)')
ax.set_title('All q values\n(log-scale count)')
ax.grid(True, alpha=0.3)

# --- Plot 2: non-zero q values only ---
ax = axes[1]
nonzero_q = all_q[all_q > 0]
ax.hist(nonzero_q, bins=80, color='darkorange', edgecolor='none')
ax.set_xlabel('q(x,y) [m/s]')
ax.set_ylabel('Count')
ax.set_title(f'Non-zero q values only\n({len(nonzero_q):,} pixels open)')
ax.grid(True, alpha=0.3)

# --- Plot 3: total volumetric flow Q per sample ---
ax = axes[2]
# Q = sum of q(x,y) * delta_A for each sample
# delta_A varies per sample, so we use the per-sample value from params
delta_A_vals = train_params['delta_A'].values          # shape [1500]
q_sums = train_labels.reshape(len(train_labels), -1).sum(axis=1)  # sum over pixels
Q_per_sample = q_sums * delta_A_vals                   # Q [m³/s] per sample
ax.hist(Q_per_sample, bins=60, color='seagreen', edgecolor='none')
ax.set_xlabel(r'$Q = \sum_{xy} q(x,y) \cdot \Delta A$ [m³/s]')
ax.set_ylabel('Count')
ax.set_title('Total volumetric flow Q per sample')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_flow_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_flow_distribution.png")

print(f"\nNon-zero q pixels: {len(nonzero_q):,} / {len(all_q):,} total ({100*len(nonzero_q)/len(all_q):.1f}% open)")
print(f"Q range: [{Q_per_sample.min():.3e}, {Q_per_sample.max():.3e}] m³/s")
print(f"Q mean:   {Q_per_sample.mean():.3e} m³/s")

### 1.4 — Open pixel fraction per sample

**What we plot:** For each of the 1500 samples, what fraction of the 32×64 = 2048 pixels are open (=1)?

**Why this matters:**
- This tells us how porous the media is on average. A sample with 10% open pixels is very dense (little flow); one with 80% is nearly empty (high flow).
- The distribution of open fractions tells us whether the training set is balanced or biased toward one type of medium.
- It also gives us intuition for the *non-physical output* problem (rubric B): a network predicting q > 0 on closed pixels is physically wrong, regardless of how small the value is.

In [ ]:
# Open fraction = number of pixels with value 1 / total pixels per sample
open_fractions = train_inputs.reshape(len(train_inputs), -1).mean(axis=1)  # shape [1500]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Open pixel fraction across training samples', fontsize=13)

# Histogram
ax = axes[0]
ax.hist(open_fractions, bins=50, color='mediumpurple', edgecolor='none')
ax.set_xlabel('Open pixel fraction (porosity)')
ax.set_ylabel('Number of samples')
ax.set_title('Distribution of porosity')
ax.axvline(open_fractions.mean(), color='red', linestyle='--', label=f'Mean = {open_fractions.mean():.2f}')
ax.legend()
ax.grid(True, alpha=0.3)

# Scatter: porosity vs total flow Q
ax = axes[1]
ax.scatter(open_fractions, Q_per_sample, alpha=0.3, s=8, color='steelblue')
ax.set_xlabel('Open pixel fraction (porosity)')
ax.set_ylabel(r'Total flow $Q$ [m³/s]')
ax.set_title('Porosity vs total volumetric flow')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/exploration_porosity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/exploration_porosity.png")

print(f"\nOpen fraction — min: {open_fractions.min():.2f}  max: {open_fractions.max():.2f}  mean: {open_fractions.mean():.2f}")

### 1.5 — Exploration summary

What we now know from the data, and how it feeds into the report:

| Observation | Report implication |
|-------------|-------------------|
| q = 0 exactly on closed pixels | Hard physical constraint → non-physical output prevention (rubric B) |
| q scales with ΔP, 1/L, 1/μ | Dimensional analysis → degree-1 homogeneous form (rubric A) |
| Cross-sections have left-right and up-down symmetry potential | Symmetry analysis → equivariance of q(x,y) (rubric A) |
| Parameters vary independently across samples | Model must generalise across all (ΔP, L, μ, ΔA) combinations |
| Porosity varies widely | Loss function must handle both near-zero and large flow magnitudes robustly |
| Input tensor is binary (0/1) — doubles as a spatial mask | Masking strategy: multiply network output by input mask in Step 5 to zero closed-pixel predictions (rubric B) |

**Next step:** Preprocessing pipeline — normalisation, train/validation split, PyTorch DataLoaders.

---
## Step 2 — Preprocessing Pipeline

**What this step does:** Compute the physical velocity scale v₀ per sample, normalize labels, split into train/val sets, and wrap everything in PyTorch Datasets and DataLoaders.

**Why v₀ normalisation?** The raw flow fields q(x,y) span many orders of magnitude depending on (ΔP, L, μ, ΔA). A network trained on raw SI values would need to learn to output values ranging from ~10⁻⁸ to ~10² m/s simultaneously — numerically unstable. The physics gives us a natural velocity scale:

$$v_0 = \frac{\Delta P \cdot \Delta A}{\mu \cdot L} \quad [\mathrm{m/s}]$$

This scaling follows from a Buckingham π dimensional analysis of the four governing parameters: ΔP [Pa] = [kg·m⁻¹·s⁻²], ΔA [m²], μ [Pa·s] = [kg·m⁻¹·s⁻¹], L [m]. The unique degree-1 combination that yields [m/s] is ΔP·ΔA/(μ·L), since [Pa·m²/(Pa·s·m)] = [m·s⁻¹]. The linear dependence on ΔP and ΔA and the inverse dependence on μ and L is also consistent with Stokes flow physics: a larger pressure drop drives faster flow, a larger pixel area represents a wider open passage carrying proportionally more flux, greater viscosity resists motion, and a longer channel offers more friction resistance. The dimensionless ratio q/v₀ is O(1) across all samples — this is verified by the statistics printed below: the max value should be of order 1, not 10⁴ or 10⁻⁴. The network learns q/v₀ from the binary cross-section; at inference we multiply back by v₀ to recover q [m/s]. This adapts the same pattern as `make_dim_hom()` in lab4, which formed dimensionless ratios of pendulum parameters.

**Split strategy:** 80/20 random split with the fixed seed → 1200 train + 300 validation samples. The 80/20 ratio is a standard choice for datasets of this size: 1200 samples provides enough diversity for a CNN to learn spatial features, while 300 validation samples gives a statistically reliable estimate of generalisation error. A 90/10 split would yield only 150 validation samples, making the val loss curve noisy and unreliable as an early-stopping signal. Split is done on raw indices before any tensor conversion to prevent data leakage.

**Batch size:** `BATCH_SIZE = 32` balances gradient quality against memory and training speed. Larger batches give more accurate (lower-variance) gradient estimates per step, but consume more GPU memory and can overshoot sharp loss minima; smaller batches introduce beneficial noise that acts as implicit regularisation. With 1200 training samples and batch size 32, each epoch runs 37–38 gradient steps — enough to observe stable loss curves without excessive memory use.

**Dataset and shapes:** Adapts `TensorData` from lab6. Inputs get a channel dimension added (shape: N×1×32×64) so the CNN sees a standard 1-channel image. Labels are kept as (N×32×64) — no channel dimension — because the ground truth is a spatial map indexed by pixel location, not an image with colour channels. The CNN will output (N×1×32×64); the loss function in Step 6 will squeeze the prediction channel dimension to (N×32×64) before comparing with the label.

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Guard: fail immediately if CSV column names don't match expectations
_expected_cols = {'delta_p', 'L', 'visc', 'delta_A'}
assert _expected_cols.issubset(train_params.columns), \
    f"Missing columns in train_params.csv. Expected {_expected_cols}, got {set(train_params.columns)}"

# ---------------------------------------------------------------------------
# 2.1 — Compute velocity scale v0 per sample
# adapted from lab4: make_dim_hom()
# changes: replaced pendulum variables with (delta_p, delta_A, visc, L)
# ---------------------------------------------------------------------------
delta_p = train_params['delta_p'].values   # ΔP [Pa],   shape (1500,)
L_vals  = train_params['L'].values         # L   [m],    shape (1500,)
mu_vals = train_params['visc'].values      # μ   [Pa·s], shape (1500,)
dA_vals = train_params['delta_A'].values   # ΔA  [m²],   shape (1500,)

v0 = delta_p * dA_vals / (mu_vals * L_vals)   # [m/s], shape (1500,)

print("=== v0 (velocity scale) statistics ===")
print(f"v0 min:  {v0.min():.4e} m/s")
print(f"v0 max:  {v0.max():.4e} m/s")
print(f"v0 mean: {v0.mean():.4e} m/s")

# ---------------------------------------------------------------------------
# 2.2 — Normalise labels by v0
# q_norm[i] = q[i] / v0[i]  →  dimensionless, O(1)
# ---------------------------------------------------------------------------
# broadcast v0 over the 32×64 spatial grid
q_norm = train_labels / v0[:, None, None]   # shape (1500, 32, 64)

print(f"\n=== Normalised q/v0 statistics ===")
print(f"q/v0 min:  {q_norm.min():.4f}")
print(f"q/v0 max:  {q_norm.max():.4f}")
print(f"q/v0 mean: {q_norm.mean():.4f}")
# Sanity check: q/v0 max should be O(1). If >> 10, the v0 formula is wrong.

# ---------------------------------------------------------------------------
# 2.3 — 80/20 train/val split (on indices, before tensor conversion)
# ---------------------------------------------------------------------------
N = len(train_inputs)                             # 1500

# Re-seed PyTorch before randperm so the split indices are always identical
# regardless of how many PyTorch random ops ran earlier in this notebook
# (e.g. accidentally re-running a training cell). This is intentional: the
# train/val partition must be stable across all runs.
torch.manual_seed(SEED)
perm = torch.randperm(N).numpy()                  # reproducible shuffle

n_train = int(0.8 * N)                            # 1200
train_idx = perm[:n_train]                        # shape (1200,)
val_idx   = perm[n_train:]                        # shape (300,)

print(f"\n=== Train/val split ===")
print(f"Train samples: {len(train_idx)}")
print(f"Val   samples: {len(val_idx)}")

# ---------------------------------------------------------------------------
# 2.4 — Convert to float32 tensors and add channel dimension
# inputs:  (N, 32, 64) → (N, 1, 32, 64)   (1-channel image for CNN)
# labels:  (N, 32, 64) — no channel dim; CNN output (N,1,32,64) will be
#          squeezed in the loss function (Step 6) before comparing
# ---------------------------------------------------------------------------
inputs_tensor = torch.tensor(train_inputs, dtype=torch.float32).unsqueeze(1)  # (1500,1,32,64)
labels_tensor = torch.tensor(q_norm,       dtype=torch.float32)               # (1500,32,64)

# Slice by split indices
X_train, y_train = inputs_tensor[train_idx], labels_tensor[train_idx]
X_val,   y_val   = inputs_tensor[val_idx],   labels_tensor[val_idx]
v0_train = torch.tensor(v0[train_idx], dtype=torch.float32)   # [m/s] kept for de-normalisation
v0_val   = torch.tensor(v0[val_idx],   dtype=torch.float32)   # [m/s] kept for de-normalisation

print(f"\nX_train shape: {X_train.shape}   y_train shape: {y_train.shape}")
print(f"X_val   shape: {X_val.shape}   y_val   shape: {y_val.shape}")

# ---------------------------------------------------------------------------
# 2.5 — PyTorch Dataset and DataLoader
# adapted from lab6: TensorData class
# changes: removed device param — tensors stay on CPU here and are moved
#          to device in the training loop via batch.to(device)
# ---------------------------------------------------------------------------
class TensorData(Dataset):
    # adapted from lab6: TensorData
    # changes: removed device param — moved to device in training loop
    def __init__(self, input_tensor, label_tensor):
        self.input  = input_tensor
        self.labels = label_tensor

    def __len__(self):
        return self.input.size(0)

    def __getitem__(self, idx):
        return self.input[idx], self.labels[idx]


BATCH_SIZE = 32

train_dataset = TensorData(X_train, y_train)
val_dataset   = TensorData(X_val,   y_val)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)   # reproducible shuffle
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

print(f"\n=== DataLoaders ===")
print(f"Train batches: {len(train_loader)}  (batch size {BATCH_SIZE})")
print(f"Val   batches: {len(val_loader)}")

# Quick sanity check: one batch
xb, yb = next(iter(train_loader))
print(f"\nSample batch — x: {xb.shape}, y: {yb.shape}")
print(f"x values in {{0,1}}: {xb.unique().tolist()}")
print(f"y (q/v0) range in batch: [{yb.min():.4f}, {yb.max():.4f}]")

---
## Step 3 — Dimensional Analysis

**What this step does:** Encapsulate the velocity scale v₀ in a reusable `make_dim_hom()` function, formally derive it from Buckingham π analysis, and verify that the second dimensionless group π₂ = ΔA/L² is constant across the training set.

**Buckingham π theorem applied to this problem:**

We have 4 physical parameters and 3 independent base units:

| Parameter | Symbol | SI units | Dimensions |
|-----------|--------|----------|------------|
| Pressure drop | ΔP | Pa | kg · m⁻¹ · s⁻² |
| Channel length | L | m | m |
| Dynamic viscosity | μ | Pa·s | kg · m⁻¹ · s⁻¹ |
| Pixel area | ΔA | m² | m² |

The output q has dimensions [m/s] = m · s⁻¹. By the Buckingham π theorem: 4 parameters + 1 output − 3 base units = **2 dimensionless groups**. One group is the target itself: π₁ = q / v₀. The second group is a dimensionless geometric ratio: π₂ = ΔA / L², which compares the pixel area to the square of the channel length. If π₂ were not constant across the dataset, it would need to be passed to the network as an additional scalar input. The code below verifies its distribution numerically. If π₂ is effectively constant (coefficient of variation < 1%), it carries no information and can be safely ignored — the network only needs the binary cross-section as input.

The velocity scale that absorbs π₁:

$$v_0 = \frac{\Delta P \cdot \Delta A}{\mu \cdot L} \quad [\text{m/s}]$$

Dimensional verification: [Pa · m² / (Pa·s · m)] = [kg·m⁻¹·s⁻² · m² / (kg·m⁻¹·s⁻¹ · m)] = [m · s⁻¹] ✓

The network learns q/v₀ from the binary cross-section; v₀ is reinjected at inference by multiplying the output by v₀. This is the `make_dim_hom()` pattern from lab4, where pendulum variables were combined into a dimensionless ratio — here we replace (g, L_pend, drop) with (ΔP, ΔA, μ, L_channel).

**Adaptation from lab4 `make_dim_hom(df)`:**
- Lab4 form: took a DataFrame, returned dimensionless pendulum features √(g/L), √(g·drop/L²)
- Here: takes arrays (delta_p, L, mu, delta_A), returns v₀ [m/s] per sample
- Key change: output is a velocity scale used to normalise the label, not a dimensionless input feature

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# 3.1 — make_dim_hom(): velocity scale from physical parameters
# adapted from lab4: make_dim_hom(df)
# changes: accepts arrays instead of a DataFrame; returns v0 [m/s] as the
#          label-normalising velocity scale rather than dimensionless inputs;
#          formula derived from Buckingham π (see markdown above)
# ---------------------------------------------------------------------------
def make_dim_hom(delta_p, L, mu, delta_A):
    """
    Compute the physical velocity scale v0 per sample.

    Derived from Buckingham pi analysis of (delta_p, L, mu, delta_A):
        v0 = delta_p * delta_A / (mu * L)   [m/s]

    The dimensionless label q/v0 is O(1) for all samples.

    Parameters
    ----------
    delta_p : array-like, shape (N,)   pressure drop [Pa]
    L       : array-like, shape (N,)   channel length [m]
    mu      : array-like, shape (N,)   dynamic viscosity [Pa·s]
    delta_A : array-like, shape (N,)   pixel area [m²]

    Returns
    -------
    v0 : np.ndarray, shape (N,)        velocity scale [m/s]
    """
    delta_p = np.asarray(delta_p, dtype=np.float64)
    L       = np.asarray(L,       dtype=np.float64)
    mu      = np.asarray(mu,      dtype=np.float64)
    delta_A = np.asarray(delta_A, dtype=np.float64)
    assert (delta_p > 0).all() and (L > 0).all() and (mu > 0).all() and (delta_A > 0).all(), \
        "All physical parameters must be strictly positive (no division-by-zero risk)"
    return delta_p * delta_A / (mu * L)   # [m/s]


# ---------------------------------------------------------------------------
# 3.2 — Verify second Buckingham π group: π₂ = ΔA / L²
# If π₂ is constant (CV < 1%), it carries no information and the network
# does not need it as an additional input.
# ---------------------------------------------------------------------------
pi2 = train_params['delta_A'].values / train_params['L'].values**2   # dimensionless

pi2_mean = pi2.mean()
pi2_std  = pi2.std()
pi2_cv   = pi2_std / pi2_mean * 100   # coefficient of variation [%]

print("=== Second Buckingham π group: π₂ = ΔA / L² ===")
print(f"min  : {pi2.min():.6e}")
print(f"max  : {pi2.max():.6e}")
print(f"mean : {pi2_mean:.6e}")
print(f"std  : {pi2_std:.6e}")
print(f"CV   : {pi2_cv:.4f}%")

if pi2_cv < 1.0:
    print(f"\nπ₂ is effectively constant (CV = {pi2_cv:.4f}% < 1%) ✓")
    print("→ π₂ carries no discriminating information across samples.")
    print("→ The network does NOT need ΔA/L² as an additional scalar input.")
    print("→ The binary cross-section + v₀ normalisation fully specifies the problem.")
else:
    print(f"\nWARNING: π₂ varies significantly (CV = {pi2_cv:.2f}% ≥ 1%).")
    print("→ π₂ = ΔA/L² should be included as an additional per-sample input scalar.")

# ---------------------------------------------------------------------------
# 3.3 — Apply make_dim_hom and verify O(1) dimensionless labels
# ---------------------------------------------------------------------------
v0_full = make_dim_hom(
    train_params['delta_p'].values,
    train_params['L'].values,
    train_params['visc'].values,
    train_params['delta_A'].values
)

# Cross-check against inline v0 from Step 2
assert np.allclose(v0_full, v0), "make_dim_hom output does not match Step 2 v0 — formula mismatch"
print("\nmake_dim_hom output matches Step 2 v0 ✓")

q_over_v0 = train_labels / v0_full[:, None, None]   # shape (1500, 32, 64)

print(f"\n=== Dimensionless field q/v0 ===")
print(f"min : {q_over_v0.min():.4f}")
print(f"max : {q_over_v0.max():.4f}   (should be O(1))")
print(f"mean: {q_over_v0.mean():.4f}")
print(f"std : {q_over_v0.std():.4f}")

# ---------------------------------------------------------------------------
# 3.4 — Verification plots: π₂ distribution + q/v₀ histogram
# ---------------------------------------------------------------------------
nonzero_mask = q_over_v0 > 0
q_over_v0_nz = q_over_v0[nonzero_mask]
per_sample_max = q_over_v0.reshape(len(train_labels), -1).max(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Dimensional analysis verification', fontsize=12)

# π₂ distribution
ax = axes[0]
ax.hist(pi2, bins=50, color='mediumpurple', edgecolor='none')
ax.set_xlabel(r'$\pi_2 = \Delta A \,/\, L^2$ (dimensionless)')
ax.set_ylabel('Count')
ax.set_title(f'Second π group distribution\nCV = {pi2_cv:.4f}%  ({"constant ✓" if pi2_cv < 1 else "VARIES — add as input"})')
ax.grid(True, alpha=0.3)

# Non-zero q/v₀ histogram
ax = axes[1]
ax.hist(q_over_v0_nz, bins=80, color='steelblue', edgecolor='none')
ax.set_xlabel(r'$q / v_0$ (dimensionless)')
ax.set_ylabel('Count')
ax.set_title(f'Non-zero q/v₀ values\n({len(q_over_v0_nz):,} open pixels)')
ax.grid(True, alpha=0.3)

# Per-sample max q/v₀
ax = axes[2]
ax.hist(per_sample_max, bins=60, color='darkorange', edgecolor='none')
ax.set_xlabel(r'max pixel $q / v_0$ per sample')
ax.set_ylabel('Number of samples')
ax.set_title(f'Per-sample max of q/v₀\nmean={per_sample_max.mean():.3f}  std={per_sample_max.std():.3f}')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/plots/dim_analysis_verification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to experiments/plots/dim_analysis_verification.png")